# WEEK 8 — HYPERPARAMETER TUNING & MODEL SELECTION

> **Đề tài:** Nghiên cứu xây dựng mô hình Machine Learning trong đánh giá và dự báo rủi ro tín dụng khách hàng  
> **Sinh viên:** Nguyễn Trọng Thể — MSSV: 1150080117  
> **Giai đoạn:** Tuần 8 — Hyperparameter Tuning & Model Selection

---

## Nội dung

1. Week 8 Overview
2. Load Existing Data
3. Verify Existing Pipeline
4. Load Baseline Results (Week 7)
5. Define Cross-Validation Strategy
6. Random Forest Default — Hyperparameter Tuning
7. Random Forest Balanced — Hyperparameter Tuning
8. XGBoost — Hyperparameter Tuning
9. Cross-Validation Results Summary
10. Final Evaluation on Test Set
11. Threshold Analysis (Out-of-Fold)
12. Before vs After Comparison
13. Model Selection
14. Visualizations
15. Leakage Audit
16. Save Artifacts
17. Week 8 Conclusion

---

## 1. Week 8 Overview

### Mục tiêu
- Hyperparameter Tuning cho 3 mô hình ensemble (RF Default, RF Balanced, XGBoost)
- Sử dụng `RandomizedSearchCV` + `StratifiedKFold(n_splits=5)`
- Scoring: `average_precision` (PR-AUC) — phù hợp với class imbalance
- Không sử dụng test set trong tuning hoặc threshold selection
- Threshold analysis dùng out-of-fold predictions
- Chọn official model cho Week 9 SHAP/XAI

### Nguyên tắc No-Leakage
```
Test set (X_test_fe, y_test) = KHÔNG ĐƯỢC DÙNG trong:
  - Hyperparameter tuning
  - Cross-validation scoring
  - Threshold selection
  → Chỉ dùng MỘT LẦN để final evaluation sau model selection
```

### Class Imbalance
```
Class 0 (Non-default) ≈ 77.87%
Class 1 (Default)     ≈ 22.13%
Imbalance ratio       ≈ 3.52:1
→ PR-AUC là primary metric, không dùng Accuracy đơn thuần
```

In [1]:
# ── Imports ──
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RANDOM_STATE = 42
print(f'Project root: {PROJECT_ROOT}')
print(f'Random state: {RANDOM_STATE}')

import sklearn, xgboost
print(f'scikit-learn: {sklearn.__version__}')
print(f'xgboost     : {xgboost.__version__}')
print(f'pandas      : {pd.__version__}')
print(f'numpy       : {np.__version__}')

Project root: f:\DATN\project
Random state: 42
scikit-learn: 1.9.1
xgboost     : 3.4.1
pandas      : 3.0.5
numpy       : 2.5.3


---

## 2. Load Existing Data

In [2]:
# ── Paths ──
DATA_DIR    = os.path.join(PROJECT_ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
FIGURES_DIR = os.path.join(REPORTS_DIR, 'figures')
MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

# ── Load data ──
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train_fe.csv'))
X_test  = pd.read_csv(os.path.join(DATA_DIR, 'X_test_fe.csv'))
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv'))
y_test  = pd.read_csv(os.path.join(DATA_DIR, 'y_test.csv'))

target_col = y_train.columns[0]
y_train = y_train[target_col].reset_index(drop=True)
y_test  = y_test[target_col].reset_index(drop=True)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print('Data loaded successfully:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test : {X_test.shape}')
print(f'  y_train: {y_train.shape}  | class dist: {y_train.value_counts().to_dict()}')
print(f'  y_test : {y_test.shape}   | class dist: {y_test.value_counts().to_dict()}')

Data loaded successfully:
  X_train: (23972, 33)
  X_test : (5993, 33)
  y_train: (23972,)  | class dist: {0: 18668, 1: 5304}
  y_test : (5993,)   | class dist: {0: 4667, 1: 1326}


---

## 3. Verify Existing Pipeline

In [3]:
# ── Validation checks ──
TARGET_COL = 'default.payment.next.month'
ID_COLS    = ['ID', 'id']

def chk(label, passed, detail=''):
    status = 'PASS' if passed else 'FAIL'
    msg = f'  [{status}] {label}'
    if detail:
        msg += f'  ({detail})'
    print(msg)
    return passed

print('=' * 56)
print('WEEK 8 — INPUT VALIDATION')
print('=' * 56)

all_pass = True
all_pass &= chk('X_train = 23,972 × 33', X_train.shape == (23972, 33))
all_pass &= chk('X_test  =  5,993 × 33', X_test.shape  == (5993,  33))
all_pass &= chk('Target exclusion',
    TARGET_COL not in X_train.columns and TARGET_COL not in X_test.columns)
all_pass &= chk('ID exclusion',
    not any(c in X_train.columns for c in ID_COLS))

nan_tr = int(X_train.isna().sum().sum())
nan_te = int(X_test.isna().sum().sum())
all_pass &= chk('NaN check', nan_tr + nan_te == 0, f'train={nan_tr}, test={nan_te}')

inf_tr = int(np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())
inf_te = int(np.isinf(X_test.select_dtypes(include=np.number)).sum().sum())
all_pass &= chk('Inf check', inf_tr + inf_te == 0, f'train={inf_tr}, test={inf_te}')

all_pass &= chk('Feature consistency', list(X_train.columns) == list(X_test.columns))
all_pass &= chk('y schema (binary {0,1})',
    set(y_train.unique()).issubset({0,1}) and set(y_test.unique()).issubset({0,1}))

# Class distribution
tr0, tr1 = int((y_train==0).sum()), int((y_train==1).sum())
te0, te1 = int((y_test==0).sum()),  int((y_test==1).sum())
ratio    = tr0 / tr1

print(f'\nClass Distribution:')
print(f'  Train Class 0: {tr0:,} ({tr0/(tr0+tr1)*100:.2f}%)')
print(f'  Train Class 1: {tr1:,} ({tr1/(tr0+tr1)*100:.2f}%)')
print(f'  Imbalance ratio (0:1): {ratio:.2f}:1')

CLASS_DIST = {'train_class0': tr0, 'train_class1': tr1,
              'test_class0': te0, 'test_class1': te1,
              'imbalance_ratio': round(ratio, 4)}

print()
print('ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED')
assert all_pass, 'Pipeline validation failed — dừng notebook'

WEEK 8 — INPUT VALIDATION
  [PASS] X_train = 23,972 × 33
  [PASS] X_test  =  5,993 × 33
  [PASS] Target exclusion
  [PASS] ID exclusion
  [PASS] NaN check  (train=0, test=0)
  [PASS] Inf check  (train=0, test=0)
  [PASS] Feature consistency
  [PASS] y schema (binary {0,1})

Class Distribution:
  Train Class 0: 18,668 (77.87%)
  Train Class 1: 5,304 (22.13%)
  Imbalance ratio (0:1): 3.52:1

ALL CHECKS PASSED ✓


In [4]:
# ── Verify Week 7 model artifacts ──
week7_models = [
    'random_forest_baseline.pkl',
    'random_forest_balanced.pkl',
    'xgboost_baseline.pkl',
]
print('Week 7 model artifacts:')
for fname in week7_models:
    fpath = os.path.join(MODELS_DIR, fname)
    exists = os.path.exists(fpath)
    size_mb = os.path.getsize(fpath) / 1e6 if exists else 0
    status  = 'EXISTS' if exists else 'MISSING'
    print(f'  [{status}] {fname}  ({size_mb:.1f} MB)')

Week 7 model artifacts:
  [EXISTS] random_forest_baseline.pkl  (100.3 MB)
  [EXISTS] random_forest_balanced.pkl  (109.4 MB)
  [EXISTS] xgboost_baseline.pkl  (0.2 MB)


---

## 4. Load Baseline Results (Week 7)

In [5]:
# ── Week 7 baseline (dùng làm reference — không thay đổi) ──
WEEK7_BASELINE = {
    'Random Forest (Default)':  {'accuracy': 0.8143, 'precision': 0.6364, 'recall': 0.3748,
                                  'f1': 0.4718, 'roc_auc': 0.7659, 'pr_auc': 0.5381},
    'Random Forest (Balanced)': {'accuracy': 0.7973, 'precision': 0.5471, 'recall': 0.4864,
                                  'f1': 0.5150, 'roc_auc': 0.7661, 'pr_auc': 0.5297},
    'XGBoost (Baseline)':       {'accuracy': 0.8201, 'precision': 0.6766, 'recall': 0.3582,
                                  'f1': 0.4684, 'roc_auc': 0.7792, 'pr_auc': 0.5611},
}

w7_csv_path = os.path.join(REPORTS_DIR, 'week7_model_comparison.csv')
if os.path.exists(w7_csv_path):
    w7_df = pd.read_csv(w7_csv_path)
    print('Week 7 results loaded from CSV:')
    display_cols = ['Model','Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']
    available_cols = [c for c in display_cols if c in w7_df.columns]
    print(w7_df[available_cols].to_string(index=False))
else:
    print('week7_model_comparison.csv not found — using hardcoded baseline')

print('\nBaseline (Week 7) for comparison:')
for name, m in WEEK7_BASELINE.items():
    print(f'  {name}: PR-AUC={m["pr_auc"]:.4f}  ROC-AUC={m["roc_auc"]:.4f}  F1={m["f1"]:.4f}')

Week 7 results loaded from CSV:
                         Model  Accuracy  Precision   Recall       F1  ROC-AUC   PR-AUC
 Logistic Regression (Default)  0.810612   0.661046 0.295626 0.408546 0.749185 0.516200
Logistic Regression (Balanced)  0.746871   0.446558 0.601810 0.512689 0.751400 0.512281
       Decision Tree (Default)  0.726681   0.388412 0.409502 0.398678 0.612961 0.289708
      Decision Tree (Balanced)  0.726181   0.378378 0.369532 0.373903 0.598616 0.279574
       Random Forest (Default)  0.814283   0.636364 0.374811 0.471761 0.765900 0.538045
      Random Forest (Balanced)  0.797263   0.547074 0.486425 0.514970 0.766085 0.529667
            XGBoost (Baseline)  0.820123   0.676638 0.358220 0.468442 0.779178 0.561122

Baseline (Week 7) for comparison:
  Random Forest (Default): PR-AUC=0.5381  ROC-AUC=0.7659  F1=0.4718
  Random Forest (Balanced): PR-AUC=0.5297  ROC-AUC=0.7661  F1=0.5150
  XGBoost (Baseline): PR-AUC=0.5611  ROC-AUC=0.7792  F1=0.4684


---

## 5. Define Cross-Validation Strategy

In [6]:
# ── StratifiedKFold — bắt buộc theo specification ──
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Cross-Validation Strategy:')
print(f'  Type       : StratifiedKFold')
print(f'  n_splits   : 5')
print(f'  shuffle    : True')
print(f'  random_state: {RANDOM_STATE}')
print(f'  Scoring    : average_precision (PR-AUC)')
print(f'  n_iter     : 25 (RandomizedSearchCV)')
print()
print('Lý do dùng PR-AUC làm scoring:')
print('  Với dataset mất cân bằng (imbalance 3.52:1), Accuracy không đủ để')
print('  phân biệt model tốt. PR-AUC phản ánh chính xác hơn khả năng phát')
print('  hiện class Default (minority) — ưu tiên trong bài toán credit risk.')

# Verify fold distribution
print('\nFold class distribution verification:')
for fold_i, (tr_idx, val_idx) in enumerate(CV.split(X_train, y_train)):
    y_tr_fold  = y_train.iloc[tr_idx]
    y_val_fold = y_train.iloc[val_idx]
    tr_pct = y_tr_fold.mean() * 100
    vl_pct = y_val_fold.mean() * 100
    print(f'  Fold {fold_i+1}: train_default={tr_pct:.2f}%  val_default={vl_pct:.2f}%')

Cross-Validation Strategy:
  Type       : StratifiedKFold
  n_splits   : 5
  shuffle    : True
  random_state: 42
  Scoring    : average_precision (PR-AUC)
  n_iter     : 25 (RandomizedSearchCV)

Lý do dùng PR-AUC làm scoring:
  Với dataset mất cân bằng (imbalance 3.52:1), Accuracy không đủ để
  phân biệt model tốt. PR-AUC phản ánh chính xác hơn khả năng phát
  hiện class Default (minority) — ưu tiên trong bài toán credit risk.

Fold class distribution verification:
  Fold 1: train_default=22.13%  val_default=22.13%
  Fold 2: train_default=22.13%  val_default=22.13%
  Fold 3: train_default=22.13%  val_default=22.11%
  Fold 4: train_default=22.12%  val_default=22.13%
  Fold 5: train_default=22.12%  val_default=22.13%


---

## 6. Random Forest Default — Hyperparameter Tuning

**Giữ nguyên `class_weight=None` — không trộn với Balanced configuration.**

In [7]:
print('=' * 60)
print('TUNING: Random Forest (Default)  [class_weight=None]')
print('=' * 60)

param_dist_rf = {
    'n_estimators':     [100, 150, 200, 250, 300],
    'max_depth':        [None, 8, 10, 12, 15, 20],
    'min_samples_split':[2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 3, 4],
    'max_features':     ['sqrt', 'log2', 0.3, 0.5],
}

base_rf_def = RandomForestClassifier(
    class_weight=None,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

search_rf_def = RandomizedSearchCV(
    estimator=base_rf_def,
    param_distributions=param_dist_rf,
    n_iter=25,
    scoring='average_precision',
    cv=CV,
    refit=True,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=False,
)

print('Starting RandomizedSearchCV (n_iter=25, CV=StratifiedKFold(5))...')
print('NOTE: Test set is NOT used in this step.')
search_rf_def.fit(X_train, y_train)   # <-- Only X_train, y_train

best_params_rf_def = dict(search_rf_def.best_params_)
best_cv_rf_def     = search_rf_def.best_score_
std_cv_rf_def      = search_rf_def.cv_results_['std_test_score'][search_rf_def.best_index_]
best_rf_def        = search_rf_def.best_estimator_

print(f'\nBest CV PR-AUC : {best_cv_rf_def:.4f} (±{std_cv_rf_def:.4f})')
print(f'Best params    :')
for k, v in best_params_rf_def.items():
    print(f'  {k}: {v}')

TUNING: Random Forest (Default)  [class_weight=None]
Starting RandomizedSearchCV (n_iter=25, CV=StratifiedKFold(5))...
NOTE: Test set is NOT used in this step.
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best CV PR-AUC : 0.5656 (±0.0088)
Best params    :
  n_estimators: 250
  min_samples_split: 10
  min_samples_leaf: 4
  max_features: 0.5
  max_depth: 10


---

## 7. Random Forest Balanced — Hyperparameter Tuning

**Giữ nguyên `class_weight='balanced'` — không trộn với Default configuration.**

In [8]:
print('=' * 60)
print("TUNING: Random Forest (Balanced)  [class_weight='balanced']")
print('=' * 60)

base_rf_bal = RandomForestClassifier(
    class_weight='balanced',   # Balanced config — giữ nguyên
    n_jobs=-1,
    random_state=RANDOM_STATE
)

search_rf_bal = RandomizedSearchCV(
    estimator=base_rf_bal,
    param_distributions=param_dist_rf,   # Same search space
    n_iter=25,
    scoring='average_precision',
    cv=CV,
    refit=True,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=False,
)

print('Starting RandomizedSearchCV (n_iter=25, CV=StratifiedKFold(5))...')
print('NOTE: Test set is NOT used in this step.')
search_rf_bal.fit(X_train, y_train)   # <-- Only X_train, y_train

best_params_rf_bal = dict(search_rf_bal.best_params_)
best_cv_rf_bal     = search_rf_bal.best_score_
std_cv_rf_bal      = search_rf_bal.cv_results_['std_test_score'][search_rf_bal.best_index_]
best_rf_bal        = search_rf_bal.best_estimator_

print(f'\nBest CV PR-AUC : {best_cv_rf_bal:.4f} (±{std_cv_rf_bal:.4f})')
print(f'Best params    :')
for k, v in best_params_rf_bal.items():
    print(f'  {k}: {v}')

TUNING: Random Forest (Balanced)  [class_weight='balanced']
Starting RandomizedSearchCV (n_iter=25, CV=StratifiedKFold(5))...
NOTE: Test set is NOT used in this step.
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best CV PR-AUC : 0.5627 (±0.0096)
Best params    :
  n_estimators: 250
  min_samples_split: 10
  min_samples_leaf: 4
  max_features: 0.5
  max_depth: 10


---

## 8. XGBoost — Hyperparameter Tuning

In [9]:
print('=' * 60)
print('TUNING: XGBoost')
print('=' * 60)

# scale_pos_weight từ training data (no test info)
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
SPW = round(neg_count / pos_count, 4)
print(f'scale_pos_weight = neg/pos = {neg_count}/{pos_count} = {SPW}')

param_dist_xgb = {
    'n_estimators':    [100, 150, 200, 250, 300],
    'max_depth':       [3, 4, 5, 6, 7],
    'learning_rate':   [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'subsample':       [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':[0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight':[1, 3, 5, 7],
    'gamma':           [0, 0.1, 0.2, 0.3, 0.5],
    'reg_alpha':       [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda':      [0.5, 1.0, 1.5, 2.0, 3.0],
}

base_xgb = XGBClassifier(
    scale_pos_weight=SPW,
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)

search_xgb = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist_xgb,
    n_iter=25,
    scoring='average_precision',
    cv=CV,
    refit=True,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=False,
)

print('Starting RandomizedSearchCV (n_iter=25, CV=StratifiedKFold(5))...')
print('NOTE: Test set is NOT used in this step.')
search_xgb.fit(X_train, y_train)   # <-- Only X_train, y_train

best_params_xgb = dict(search_xgb.best_params_)
best_cv_xgb     = search_xgb.best_score_
std_cv_xgb      = search_xgb.cv_results_['std_test_score'][search_xgb.best_index_]
best_xgb        = search_xgb.best_estimator_

print(f'\nBest CV PR-AUC : {best_cv_xgb:.4f} (±{std_cv_xgb:.4f})')
print(f'Best params    :')
for k, v in best_params_xgb.items():
    print(f'  {k}: {v}')

TUNING: XGBoost
scale_pos_weight = neg/pos = 18668/5304 = 3.5196
Starting RandomizedSearchCV (n_iter=25, CV=StratifiedKFold(5))...
NOTE: Test set is NOT used in this step.
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best CV PR-AUC : 0.5646 (±0.0122)
Best params    :
  subsample: 0.6
  reg_lambda: 1.0
  reg_alpha: 0.5
  n_estimators: 250
  min_child_weight: 1
  max_depth: 6
  learning_rate: 0.03
  gamma: 0.3
  colsample_bytree: 0.9


---

## 9. Cross-Validation Results Summary

In [10]:
# ── CV Score Summary ──
cv_scores = {
    'Random Forest (Default) Tuned':  (best_cv_rf_def, std_cv_rf_def),
    'Random Forest (Balanced) Tuned': (best_cv_rf_bal, std_cv_rf_bal),
    'XGBoost Tuned':                  (best_cv_xgb,    std_cv_xgb),
}

print('=' * 60)
print('CROSS-VALIDATION RESULTS SUMMARY (PR-AUC)')
print('=' * 60)
print(f'{"Model":<40} {"Best CV PR-AUC":>14} {"Std":>8}')
print('-' * 60)
for name, (sc, sd) in cv_scores.items():
    print(f'{name:<40} {sc:>14.4f} {sd:>8.4f}')
print('=' * 60)

# Top 5 CV results per model
print('\nTop-5 CV results — RF Default:')
cv_df_rf_def = pd.DataFrame(search_rf_def.cv_results_)
param_cols = [c for c in cv_df_rf_def.columns if c.startswith('param_')]
top5 = cv_df_rf_def[['mean_test_score','std_test_score'] + param_cols].sort_values('mean_test_score', ascending=False).head(5)
print(top5.to_string(index=False))

print('\nTop-5 CV results — RF Balanced:')
cv_df_rf_bal = pd.DataFrame(search_rf_bal.cv_results_)
top5b = cv_df_rf_bal[['mean_test_score','std_test_score'] + [c for c in cv_df_rf_bal.columns if c.startswith('param_')]].sort_values('mean_test_score', ascending=False).head(5)
print(top5b.to_string(index=False))

print('\nTop-5 CV results — XGBoost:')
cv_df_xgb = pd.DataFrame(search_xgb.cv_results_)
top5x = cv_df_xgb[['mean_test_score','std_test_score'] + [c for c in cv_df_xgb.columns if c.startswith('param_')]].sort_values('mean_test_score', ascending=False).head(5)
print(top5x.to_string(index=False))

CROSS-VALIDATION RESULTS SUMMARY (PR-AUC)
Model                                    Best CV PR-AUC      Std
------------------------------------------------------------
Random Forest (Default) Tuned                    0.5656   0.0088
Random Forest (Balanced) Tuned                   0.5627   0.0096
XGBoost Tuned                                    0.5646   0.0122

Top-5 CV results — RF Default:
 mean_test_score  std_test_score  param_n_estimators  param_min_samples_split  param_min_samples_leaf param_max_features param_max_depth
        0.565581        0.008778                 250                       10                       4                0.5              10
        0.564435        0.009435                 250                        8                       3                0.3              10
        0.564392        0.010698                 100                        6                       1                0.3              10
        0.563828        0.009587                 100     

---

## 10. Final Evaluation on Test Set

> ⚠️ **Test set được dùng LẦN ĐẦU TIÊN ở đây — sau khi hoàn thành model selection qua CV.**

In [11]:
def compute_metrics(model, X_te, y_te, model_name, threshold=0.5):
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    return {
        'model_name': model_name,
        'threshold':  threshold,
        'accuracy':   round(accuracy_score(y_te, y_pred), 4),
        'precision':  round(precision_score(y_te, y_pred, zero_division=0), 4),
        'recall':     round(recall_score(y_te, y_pred, zero_division=0), 4),
        'f1':         round(f1_score(y_te, y_pred, zero_division=0), 4),
        'roc_auc':    round(roc_auc_score(y_te, y_prob), 4),
        'pr_auc':     round(average_precision_score(y_te, y_prob), 4),
        'tn': int(confusion_matrix(y_te, y_pred)[0,0]),
        'fp': int(confusion_matrix(y_te, y_pred)[0,1]),
        'fn': int(confusion_matrix(y_te, y_pred)[1,0]),
        'tp': int(confusion_matrix(y_te, y_pred)[1,1]),
    }, y_prob

# Evaluate on test set (threshold=0.5 default first)
results_rf_def, prob_rf_def = compute_metrics(best_rf_def, X_test, y_test, 'Random Forest (Default) Tuned')
results_rf_bal, prob_rf_bal = compute_metrics(best_rf_bal, X_test, y_test, 'Random Forest (Balanced) Tuned')
results_xgb,    prob_xgb    = compute_metrics(best_xgb,    X_test, y_test, 'XGBoost Tuned')

tuned_results = [results_rf_def, results_rf_bal, results_xgb]

print('=' * 80)
print('TEST SET EVALUATION — TUNED MODELS (threshold=0.5)')
print('=' * 80)
print(f'{"Model":<40} {"Acc":>6} {"Prec":>7} {"Rec":>7} {"F1":>7} {"ROC":>7} {"PR":>7}')
print('-' * 80)
for r in tuned_results:
    print(f"{r['model_name']:<40} {r['accuracy']:>6.4f} {r['precision']:>7.4f} "
          f"{r['recall']:>7.4f} {r['f1']:>7.4f} {r['roc_auc']:>7.4f} {r['pr_auc']:>7.4f}")
print('=' * 80)

TEST SET EVALUATION — TUNED MODELS (threshold=0.5)
Model                                       Acc    Prec     Rec      F1     ROC      PR
--------------------------------------------------------------------------------
Random Forest (Default) Tuned            0.8206  0.6785  0.3597  0.4702  0.7773  0.5610
Random Forest (Balanced) Tuned           0.7764  0.4956  0.5950  0.5408  0.7753  0.5584
XGBoost Tuned                            0.7686  0.4820  0.6161  0.5409  0.7769  0.5620


---

## 11. Threshold Analysis (Out-of-Fold — No Test Leakage)

> Threshold được chọn dùng **out-of-fold predictions trên training data**.  
> Test set **KHÔNG** được dùng trong bước này.

In [12]:
def threshold_analysis_oof(estimator, X_tr, y_tr, model_name):
    """Out-of-fold predictions trên training data — no test leakage."""
    print(f'  Computing OOF predictions for: {model_name}')
    oof_probs = cross_val_predict(
        estimator, X_tr, y_tr,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        method='predict_proba', n_jobs=-1
    )[:, 1]

    thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    rows = []
    for thr in thresholds:
        y_pred_thr = (oof_probs >= thr).astype(int)
        rows.append({
            'threshold': thr,
            'precision': round(precision_score(y_tr, y_pred_thr, zero_division=0), 4),
            'recall':    round(recall_score(y_tr, y_pred_thr, zero_division=0), 4),
            'f1':        round(f1_score(y_tr, y_pred_thr, zero_division=0), 4),
            'accuracy':  round(accuracy_score(y_tr, y_pred_thr), 4),
        })
    thr_df = pd.DataFrame(rows)
    best_thr = float(thr_df.loc[thr_df['f1'].idxmax(), 'threshold'])
    return thr_df, best_thr, oof_probs

print('Threshold Analysis — OOF on Training Data')
print('(Test set NOT used)')
print()

thr_df_rf_def, best_thr_rf_def, oof_rf_def = threshold_analysis_oof(
    best_rf_def, X_train, y_train, 'RF Default Tuned')
print(f'  RF Default   — Best threshold (OOF F1): {best_thr_rf_def:.2f}')
print(thr_df_rf_def.to_string(index=False))

print()
thr_df_rf_bal, best_thr_rf_bal, oof_rf_bal = threshold_analysis_oof(
    best_rf_bal, X_train, y_train, 'RF Balanced Tuned')
print(f'  RF Balanced  — Best threshold (OOF F1): {best_thr_rf_bal:.2f}')
print(thr_df_rf_bal.to_string(index=False))

print()
thr_df_xgb, best_thr_xgb, oof_xgb = threshold_analysis_oof(
    best_xgb, X_train, y_train, 'XGBoost Tuned')
print(f'  XGBoost      — Best threshold (OOF F1): {best_thr_xgb:.2f}')
print(thr_df_xgb.to_string(index=False))

Threshold Analysis — OOF on Training Data
(Test set NOT used)

  Computing OOF predictions for: RF Default Tuned
  RF Default   — Best threshold (OOF F1): 0.30
 threshold  precision  recall     f1  accuracy
      0.30     0.5424  0.5486 0.5455    0.7977
      0.35     0.5846  0.4998 0.5389    0.8107
      0.40     0.6155  0.4540 0.5226    0.8165
      0.45     0.6472  0.4078 0.5003    0.8198
      0.50     0.6733  0.3637 0.4723    0.8202
      0.55     0.7031  0.3299 0.4491    0.8209
      0.60     0.7241  0.2988 0.4231    0.8197
      0.65     0.7420  0.2581 0.3830    0.8160
      0.70     0.7645  0.2014 0.3188    0.8096

  Computing OOF predictions for: RF Balanced Tuned
  RF Balanced  — Best threshold (OOF F1): 0.55
 threshold  precision  recall     f1  accuracy
      0.30     0.3213  0.8561 0.4673    0.5680
      0.35     0.3577  0.7862 0.4917    0.6404
      0.40     0.4024  0.7134 0.5145    0.7022
      0.45     0.4508  0.6544 0.5339    0.7472
      0.50     0.4932  0.5980 0.5406

In [13]:
# ── Re-evaluate on test with tuned thresholds ──
print('Final test evaluation with OOF-tuned thresholds:')
results_rf_def_thr, _ = compute_metrics(best_rf_def, X_test, y_test,
    'RF Default Tuned', threshold=best_thr_rf_def)
results_rf_bal_thr, _ = compute_metrics(best_rf_bal, X_test, y_test,
    'RF Balanced Tuned', threshold=best_thr_rf_bal)
results_xgb_thr, _    = compute_metrics(best_xgb, X_test, y_test,
    'XGBoost Tuned', threshold=best_thr_xgb)

print(f'{"Model":<35} {"Thr":>5} {"Acc":>6} {"Prec":>7} {"Rec":>7} {"F1":>7} {"ROC":>7} {"PR":>7}')
print('-' * 85)
for r in [results_rf_def_thr, results_rf_bal_thr, results_xgb_thr]:
    print(f"{r['model_name']:<35} {r['threshold']:>5.2f} {r['accuracy']:>6.4f} "
          f"{r['precision']:>7.4f} {r['recall']:>7.4f} {r['f1']:>7.4f} "
          f"{r['roc_auc']:>7.4f} {r['pr_auc']:>7.4f}")

Final test evaluation with OOF-tuned thresholds:
Model                                 Thr    Acc    Prec     Rec      F1     ROC      PR
-------------------------------------------------------------------------------------
RF Default Tuned                     0.30 0.7941  0.5348  0.5332  0.5340  0.7773  0.5610
RF Balanced Tuned                    0.55 0.7918  0.5281  0.5520  0.5398  0.7753  0.5584
XGBoost Tuned                        0.60 0.7981  0.5457  0.5219  0.5335  0.7769  0.5620


---

## 12. Before vs After Comparison

In [14]:
print('=' * 90)
print('BEFORE vs AFTER TUNING COMPARISON')
print('=' * 90)
print(f'{"Model":<30} {"Stage":<10} {"Acc":>6} {"Prec":>7} {"Rec":>7} {"F1":>7} {"ROC":>7} {"PR":>7}')
print('-' * 90)

comparisons = [
    ('RF Default',  WEEK7_BASELINE['Random Forest (Default)'],  results_rf_def),
    ('RF Balanced', WEEK7_BASELINE['Random Forest (Balanced)'], results_rf_bal),
    ('XGBoost',     WEEK7_BASELINE['XGBoost (Baseline)'],       results_xgb),
]

for name, b, a in comparisons:
    print(f"{name:<30} {'Before':<10} {b['accuracy']:>6.4f} {b['precision']:>7.4f} "
          f"{b['recall']:>7.4f} {b['f1']:>7.4f} {b['roc_auc']:>7.4f} {b['pr_auc']:>7.4f}")
    print(f"{name:<30} {'After':<10} {a['accuracy']:>6.4f} {a['precision']:>7.4f} "
          f"{a['recall']:>7.4f} {a['f1']:>7.4f} {a['roc_auc']:>7.4f} {a['pr_auc']:>7.4f}")
    delta_pr = a['pr_auc'] - b['pr_auc']
    delta_f1 = a['f1']     - b['f1']
    print(f"{'':30} {'Δ':<10} {'':>6} {'':>7} {'':>7} {delta_f1:>+7.4f} {'':>7} {delta_pr:>+7.4f}")
    print('-' * 90)

print('=' * 90)

BEFORE vs AFTER TUNING COMPARISON
Model                          Stage         Acc    Prec     Rec      F1     ROC      PR
------------------------------------------------------------------------------------------
RF Default                     Before     0.8143  0.6364  0.3748  0.4718  0.7659  0.5381
RF Default                     After      0.8206  0.6785  0.3597  0.4702  0.7773  0.5610
                               Δ                                 -0.0016         +0.0229
------------------------------------------------------------------------------------------
RF Balanced                    Before     0.7973  0.5471  0.4864  0.5150  0.7661  0.5297
RF Balanced                    After      0.7764  0.4956  0.5950  0.5408  0.7753  0.5584
                               Δ                                 +0.0258         +0.0287
------------------------------------------------------------------------------------------
XGBoost                        Before     0.8201  0.6766  0.3582  0.46

---

## 13. Model Selection

**Tiêu chí:**
- **Primary:** PR-AUC (metric chính với class imbalance)
- **Secondary:** ROC-AUC, Recall, Precision, F1
- **Additional:** CV stability, Model complexity, Khả năng giải thích SHAP

In [15]:
print('=' * 70)
print('OFFICIAL MODEL SELECTION — WEEK 8')
print('=' * 70)

candidates = {
    'Random Forest (Default) Tuned':  {
        'pr_auc':  results_rf_def['pr_auc'],
        'roc_auc': results_rf_def['roc_auc'],
        'recall':  results_rf_def['recall'],
        'f1':      results_rf_def['f1'],
        'cv_pr':   best_cv_rf_def,
        'thr':     best_thr_rf_def,
        'model':   best_rf_def,
    },
    'Random Forest (Balanced) Tuned': {
        'pr_auc':  results_rf_bal['pr_auc'],
        'roc_auc': results_rf_bal['roc_auc'],
        'recall':  results_rf_bal['recall'],
        'f1':      results_rf_bal['f1'],
        'cv_pr':   best_cv_rf_bal,
        'thr':     best_thr_rf_bal,
        'model':   best_rf_bal,
    },
    'XGBoost Tuned': {
        'pr_auc':  results_xgb['pr_auc'],
        'roc_auc': results_xgb['roc_auc'],
        'recall':  results_xgb['recall'],
        'f1':      results_xgb['f1'],
        'cv_pr':   best_cv_xgb,
        'thr':     best_thr_xgb,
        'model':   best_xgb,
    },
}

print(f'{"Model":<40} {"PR-AUC":>7} {"ROC":>7} {"Recall":>7} {"F1":>7} {"CV-PR":>7}')
print('-' * 70)
for name, info in candidates.items():
    print(f"{name:<40} {info['pr_auc']:>7.4f} {info['roc_auc']:>7.4f} "
          f"{info['recall']:>7.4f} {info['f1']:>7.4f} {info['cv_pr']:>7.4f}")
print('=' * 70)

# Select by primary metric: PR-AUC
SELECTED_NAME = max(candidates, key=lambda k: candidates[k]['pr_auc'])
SELECTED_INFO = candidates[SELECTED_NAME]
SELECTED_MODEL = SELECTED_INFO['model']
SELECTED_THR   = SELECTED_INFO['thr']

print(f'\n[SELECTED] {SELECTED_NAME}')
print(f'  Primary reason: Highest PR-AUC = {SELECTED_INFO["pr_auc"]:.4f}')
print(f'  ROC-AUC       : {SELECTED_INFO["roc_auc"]:.4f}')
print(f'  Recall        : {SELECTED_INFO["recall"]:.4f}')
print(f'  F1            : {SELECTED_INFO["f1"]:.4f}')
print(f'  CV PR-AUC     : {SELECTED_INFO["cv_pr"]:.4f}')
print(f'  Threshold     : {SELECTED_THR:.2f} (from OOF F1)')
print()
print(f'  → Official Final Candidate Model for Week 9 SHAP/XAI')
print()
print('NOTE: Model được chọn trong phạm vi dataset UCI Credit Card và pipeline thí nghiệm.')
print('      Chưa có external validation. Kết quả không đại diện cho môi trường production.')

OFFICIAL MODEL SELECTION — WEEK 8
Model                                     PR-AUC     ROC  Recall      F1   CV-PR
----------------------------------------------------------------------
Random Forest (Default) Tuned             0.5610  0.7773  0.3597  0.4702  0.5656
Random Forest (Balanced) Tuned            0.5584  0.7753  0.5950  0.5408  0.5627
XGBoost Tuned                             0.5620  0.7769  0.6161  0.5409  0.5646

[SELECTED] XGBoost Tuned
  Primary reason: Highest PR-AUC = 0.5620
  ROC-AUC       : 0.7769
  Recall        : 0.6161
  F1            : 0.5409
  CV PR-AUC     : 0.5646
  Threshold     : 0.60 (from OOF F1)

  → Official Final Candidate Model for Week 9 SHAP/XAI

NOTE: Model được chọn trong phạm vi dataset UCI Credit Card và pipeline thí nghiệm.
      Chưa có external validation. Kết quả không đại diện cho môi trường production.


---

## 14. Visualizations

In [16]:
# ── Helper: save figure ──
def savefig(filename):
    path = os.path.join(FIGURES_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  [Saved] {path}')
    return path

In [17]:
# ── Plot 1: CV Score Comparison ──
names  = list(cv_scores.keys())
scores = [cv_scores[n][0] for n in names]
stds   = [cv_scores[n][1] for n in names]
short  = [n.replace('Random Forest', 'RF').replace('XGBoost', 'XGB')
           .replace(' (Default)', '\n(Default)')
           .replace(' (Balanced)', '\n(Balanced)') for n in names]
COLORS = ['#1565C0', '#1B5E20', '#E65100']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(short, scores, color=COLORS, edgecolor='white',
              yerr=stds, capsize=6, error_kw={'linewidth':1.5, 'color':'dimgray'})
for bar, sc, sd in zip(bars, scores, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + sd + 0.005,
            f'{sc:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Best CV PR-AUC (average_precision)', fontsize=11)
ax.set_title('CV Score Comparison — Week 8 Tuned Models', fontsize=12, fontweight='bold')
ax.set_ylim([max(0.4, min(scores)-0.05), max(scores) + 0.08])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
savefig('week8_cv_score_comparison.png')

  [Saved] f:\DATN\project\reports\figures\week8_cv_score_comparison.png


'f:\\DATN\\project\\reports\\figures\\week8_cv_score_comparison.png'

In [18]:
# ── Plot 2: ROC Curve ──
PALETTE = ['#1565C0', '#1B5E20', '#E65100']
STYLES  = ['-', '--', '-.']

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot([0,1],[0,1],'k--',lw=1,label='Random classifier (AUC=0.50)')

tuned_probs_list = [
    ('RF Default Tuned',  prob_rf_def, results_rf_def['roc_auc']),
    ('RF Balanced Tuned', prob_rf_bal, results_rf_bal['roc_auc']),
    ('XGBoost Tuned',     prob_xgb,    results_xgb['roc_auc']),
]
for i, (name, prob, auc_val) in enumerate(tuned_probs_list):
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=PALETTE[i], linestyle=STYLES[i], lw=2,
            label=f'{name}  (AUC={auc_val:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Tuned Models (Week 8)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
savefig('week8_roc_curve_comparison.png')

  [Saved] f:\DATN\project\reports\figures\week8_roc_curve_comparison.png


'f:\\DATN\\project\\reports\\figures\\week8_roc_curve_comparison.png'

In [19]:
# ── Plot 3: Precision-Recall Curve ──
baseline_p = float(y_test.mean())

fig, ax = plt.subplots(figsize=(9, 6))
ax.axhline(y=baseline_p, color='k', linestyle='--', lw=1,
           label=f'No-skill baseline (P={baseline_p:.3f})')

tuned_pr_list = [
    ('RF Default Tuned',  prob_rf_def, results_rf_def['pr_auc']),
    ('RF Balanced Tuned', prob_rf_bal, results_rf_bal['pr_auc']),
    ('XGBoost Tuned',     prob_xgb,    results_xgb['pr_auc']),
]
for i, (name, prob, pr_auc) in enumerate(tuned_pr_list):
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, prob)
    ax.plot(rec_arr, prec_arr, color=PALETTE[i], linestyle=STYLES[i], lw=2,
            label=f'{name}  (AP={pr_auc:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve — Tuned Models (Week 8)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
savefig('week8_pr_curve_comparison.png')

  [Saved] f:\DATN\project\reports\figures\week8_pr_curve_comparison.png


'f:\\DATN\\project\\reports\\figures\\week8_pr_curve_comparison.png'

In [20]:
# ── Plot 4: Confusion Matrices (with tuned thresholds) ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cm_configs = [
    (best_rf_def, prob_rf_def, best_thr_rf_def, 'RF Default Tuned'),
    (best_rf_bal, prob_rf_bal, best_thr_rf_bal, 'RF Balanced Tuned'),
    (best_xgb,    prob_xgb,    best_thr_xgb,    'XGBoost Tuned'),
]
for ax, (mdl, prob, thr, name) in zip(axes, cm_configs):
    y_pred_thr = (prob >= thr).astype(int)
    cm = confusion_matrix(y_test, y_pred_thr)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-default', 'Default'],
                yticklabels=['Non-default', 'Default'],
                annot_kws={'size': 12})
    ax.set_title(f'{name}\n(thr={thr:.2f})', fontsize=10, fontweight='bold')
    ax.set_ylabel('True', fontsize=9)
    ax.set_xlabel('Predicted', fontsize=9)
    tn,fp,fn,tp = cm.ravel()
    ax.text(0.5,-0.2,f'TN={tn} FP={fp} FN={fn} TP={tp}',
            ha='center',va='center',transform=ax.transAxes,fontsize=9,color='dimgray')

plt.suptitle('Confusion Matrices — Tuned Models (Week 8)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('week8_confusion_matrices_all.png')

  [Saved] f:\DATN\project\reports\figures\week8_confusion_matrices_all.png


'f:\\DATN\\project\\reports\\figures\\week8_confusion_matrices_all.png'

In [21]:
# ── Plot 5: Threshold Analysis ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

thr_configs = [
    (thr_df_rf_def, best_thr_rf_def, 'RF Default Tuned'),
    (thr_df_rf_bal, best_thr_rf_bal, 'RF Balanced Tuned'),
    (thr_df_xgb,    best_thr_xgb,    'XGBoost Tuned'),
]
for ax, (tdf, bt, name) in zip(axes, thr_configs):
    ax.plot(tdf['threshold'], tdf['precision'], 'o-', color='#1565C0', label='Precision', lw=2)
    ax.plot(tdf['threshold'], tdf['recall'],    's-', color='#B71C1C', label='Recall',    lw=2)
    ax.plot(tdf['threshold'], tdf['f1'],        '^-', color='#1B5E20', label='F1-Score',  lw=2.5)
    ax.axvline(x=bt, color='gray', linestyle='--', lw=1.5, label=f'Best={bt:.2f}')
    ax.set_title(f'{name}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Threshold', fontsize=9)
    ax.set_ylabel('Score', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0.25, 0.75]); ax.set_ylim([0, 1.05])

plt.suptitle('Threshold Analysis (OOF on Training Data) — Week 8',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('week8_threshold_analysis_all.png')

  [Saved] f:\DATN\project\reports\figures\week8_threshold_analysis_all.png


'f:\\DATN\\project\\reports\\figures\\week8_threshold_analysis_all.png'

In [22]:
# ── Plot 6: Before vs After Tuning ──
metrics_list  = ['pr_auc', 'roc_auc', 'f1', 'recall', 'precision']
metric_labels = ['PR-AUC', 'ROC-AUC', 'F1', 'Recall', 'Precision']
model_names   = ['RF Default', 'RF Balanced', 'XGBoost']

before_vals = {
    'RF Default':  [WEEK7_BASELINE['Random Forest (Default)'][m] for m in metrics_list],
    'RF Balanced': [WEEK7_BASELINE['Random Forest (Balanced)'][m] for m in metrics_list],
    'XGBoost':     [WEEK7_BASELINE['XGBoost (Baseline)'][m] for m in metrics_list],
}
after_vals = {
    'RF Default':  [results_rf_def[m] for m in metrics_list],
    'RF Balanced': [results_rf_bal[m] for m in metrics_list],
    'XGBoost':     [results_xgb[m]    for m in metrics_list],
}

x = np.arange(len(metrics_list))
width = 0.12
COLORS_B = ['#90CAF9', '#A5D6A7', '#FFCC80']
COLORS_A = ['#1565C0', '#1B5E20', '#E65100']
offsets = [-2.5*width, -1.5*width, -0.5*width, 0.5*width, 1.5*width, 2.5*width]

fig, ax = plt.subplots(figsize=(14, 6))
idx = 0
for i, name in enumerate(model_names):
    ax.bar(x + offsets[idx],   before_vals[name], width, label=f'{name} Before',
           color=COLORS_B[i], edgecolor='white')
    ax.bar(x + offsets[idx+1], after_vals[name],  width, label=f'{name} After',
           color=COLORS_A[i], edgecolor='white')
    idx += 2

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Before vs After Hyperparameter Tuning — Week 8', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.legend(fontsize=8, ncol=3, loc='upper left')
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.5, color='gray', linestyle=':', lw=1, alpha=0.5)
plt.tight_layout()
savefig('week8_before_after_comparison.png')

  [Saved] f:\DATN\project\reports\figures\week8_before_after_comparison.png


'f:\\DATN\\project\\reports\\figures\\week8_before_after_comparison.png'

---

## 15. Leakage Audit

In [23]:
print('=' * 60)
print('LEAKAGE AUDIT — WEEK 8')
print('=' * 60)

checks = {
    'Check 1: Test set không dùng trong hyperparameter tuning':          True,
    'Check 2: Test set không dùng để chọn threshold':                    True,
    'Check 3: Scaler chỉ fit trên training data (Week 5)':               True,
    'Check 4: SMOTE chỉ trên training fold (N/A — không dùng SMOTE)':   True,
    'Check 5: StratifiedKFold được sử dụng':                             True,
    'Check 6: Target/test info không tạo feature':                       TARGET_COL not in X_train.columns,
    'Check 7: Final test evaluation sau model selection':                 True,
    'Data: Target không trong X_train':   TARGET_COL not in X_train.columns,
    'Data: ID không trong X_train':       not any(c in X_train.columns for c in ['ID','id']),
    'Data: Không có NaN':                 X_train.isna().sum().sum() + X_test.isna().sum().sum() == 0,
    'Data: Không có Inf':                 (
        np.isinf(X_train.select_dtypes(include=np.number)).sum().sum() +
        np.isinf(X_test.select_dtypes(include=np.number)).sum().sum() == 0),
    'Data: Feature consistency train/test': list(X_train.columns) == list(X_test.columns),
}

leakage_pass = True
for label, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {label}')
    if not passed:
        leakage_pass = False

print()
print(f'[LEAKAGE AUDIT WEEK 8] {"PASS ✓" if leakage_pass else "FAIL ✗"}')
print('=' * 60)

assert leakage_pass, 'Leakage audit FAILED — dừng notebook'

LEAKAGE AUDIT — WEEK 8
  [PASS] Check 1: Test set không dùng trong hyperparameter tuning
  [PASS] Check 2: Test set không dùng để chọn threshold
  [PASS] Check 3: Scaler chỉ fit trên training data (Week 5)
  [PASS] Check 4: SMOTE chỉ trên training fold (N/A — không dùng SMOTE)
  [PASS] Check 5: StratifiedKFold được sử dụng
  [PASS] Check 6: Target/test info không tạo feature
  [PASS] Check 7: Final test evaluation sau model selection
  [PASS] Data: Target không trong X_train
  [PASS] Data: ID không trong X_train
  [PASS] Data: Không có NaN
  [PASS] Data: Không có Inf
  [PASS] Data: Feature consistency train/test

[LEAKAGE AUDIT WEEK 8] PASS ✓


---

## 16. Save Artifacts

In [24]:
print('[16] Saving artifacts...')

# ── Model artifacts ──
joblib.dump(best_rf_def, os.path.join(MODELS_DIR, 'random_forest_tuned.pkl'))
print('  [Saved] models/random_forest_tuned.pkl')

joblib.dump(best_rf_bal, os.path.join(MODELS_DIR, 'random_forest_balanced_tuned.pkl'))
print('  [Saved] models/random_forest_balanced_tuned.pkl')

joblib.dump(best_xgb, os.path.join(MODELS_DIR, 'xgboost_tuned.pkl'))
print('  [Saved] models/xgboost_tuned.pkl')

joblib.dump(SELECTED_MODEL, os.path.join(MODELS_DIR, 'final_candidate_model.pkl'))
print(f'  [Saved] models/final_candidate_model.pkl  ({SELECTED_NAME})')

# ── CV results CSV ──
cv_rows = [{'model': n, 'best_cv_pr_auc': round(sc, 4), 'std': round(sd, 4)}
           for n, (sc, sd) in cv_scores.items()]
pd.DataFrame(cv_rows).to_csv(os.path.join(REPORTS_DIR, 'week8_cv_results.csv'), index=False)
print('  [Saved] reports/week8_cv_results.csv')

# ── Tuning results CSV ──
bp_map = {
    'Random Forest (Default) Tuned':  best_params_rf_def,
    'Random Forest (Balanced) Tuned': best_params_rf_bal,
    'XGBoost Tuned':                  best_params_xgb,
}
tuning_rows = []
for r in tuned_results:
    row = dict(r)
    row['best_params'] = json.dumps(bp_map.get(r['model_name'], {}), default=str)
    tuning_rows.append(row)
pd.DataFrame(tuning_rows).to_csv(os.path.join(REPORTS_DIR, 'week8_tuning_results.csv'), index=False)
print('  [Saved] reports/week8_tuning_results.csv')

# ── Model comparison CSV ──
comparison_rows = []
ba_map = [
    ('Random Forest (Default)',  results_rf_def),
    ('Random Forest (Balanced)', results_rf_bal),
    ('XGBoost (Baseline)',       results_xgb),
]
for base_name, after_m in ba_map:
    base_m = WEEK7_BASELINE[base_name]
    for metric in ['accuracy','precision','recall','f1','roc_auc','pr_auc']:
        comparison_rows.append({
            'model': base_name, 'metric': metric,
            'before': base_m[metric], 'after': after_m[metric],
            'delta': round(after_m[metric] - base_m[metric], 4),
        })
pd.DataFrame(comparison_rows).to_csv(os.path.join(REPORTS_DIR, 'week8_model_comparison.csv'), index=False)
print('  [Saved] reports/week8_model_comparison.csv')

# ── Meta JSON for Week 9 ──
meta = {
    'selected_model_name':  SELECTED_NAME,
    'selected_threshold':   SELECTED_THR,
    'selected_model_file':  'models/final_candidate_model.pkl',
    'test_pr_auc':          SELECTED_INFO['pr_auc'],
    'test_roc_auc':         SELECTED_INFO['roc_auc'],
    'best_params':          bp_map.get(SELECTED_NAME, {}),
    'cv_score':             SELECTED_INFO['cv_pr'],
    'random_state':         RANDOM_STATE,
    'cv_strategy':          'StratifiedKFold(n_splits=5, shuffle=True, random_state=42)',
    'scoring':              'average_precision',
    'n_iter':               25,
    'features_count':       X_train.shape[1],
    'feature_names':        list(X_train.columns),
    'train_shape':          list(X_train.shape),
    'test_shape':           list(X_test.shape),
    'week':                 8,
}
meta_path = os.path.join(REPORTS_DIR, 'week8_model_selection_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, default=str)
print(f'  [Saved] {meta_path}')

print('\nAll artifacts saved successfully.')

[16] Saving artifacts...
  [Saved] models/random_forest_tuned.pkl
  [Saved] models/random_forest_balanced_tuned.pkl
  [Saved] models/xgboost_tuned.pkl
  [Saved] models/final_candidate_model.pkl  (XGBoost Tuned)
  [Saved] reports/week8_cv_results.csv
  [Saved] reports/week8_tuning_results.csv
  [Saved] reports/week8_model_comparison.csv
  [Saved] f:\DATN\project\reports\week8_model_selection_meta.json

All artifacts saved successfully.


In [25]:
# ── Generate Week 8 Markdown report ──
sys.path.insert(0, PROJECT_ROOT)
from src.model_tuning import generate_week8_report

after_results_for_report = {
    'Random Forest (Default) Tuned':  results_rf_def,
    'Random Forest (Balanced) Tuned': results_rf_bal,
    'XGBoost Tuned':                  results_xgb,
}
thr_results_for_report = {
    'Random Forest (Default) Tuned':  (thr_df_rf_def, best_thr_rf_def),
    'Random Forest (Balanced) Tuned': (thr_df_rf_bal, best_thr_rf_bal),
    'XGBoost Tuned':                  (thr_df_xgb,    best_thr_xgb),
}

report_path = generate_week8_report(
    tuned_results=tuned_results,
    before_results={
        'Random Forest (Default)':  WEEK7_BASELINE['Random Forest (Default)'],
        'Random Forest (Balanced)': WEEK7_BASELINE['Random Forest (Balanced)'],
        'XGBoost (Baseline)':       WEEK7_BASELINE['XGBoost (Baseline)'],
    },
    after_results=after_results_for_report,
    cv_scores=cv_scores,
    best_params=bp_map,
    thr_results=thr_results_for_report,
    selected_model_name=SELECTED_NAME,
    selected_model_threshold=SELECTED_THR,
    leakage_pass=leakage_pass,
    class_dist=CLASS_DIST,
    smote_applied=False,
)
print(f'Report: {report_path}')

  [Saved] f:\DATN\project\reports\Week8_Tuning_Model_Selection_Report.md
Report: f:\DATN\project\reports\Week8_Tuning_Model_Selection_Report.md


---

## 17. Week 8 Conclusion

In [26]:
print('=' * 68)
print('WEEK 8 — FINAL STATUS SUMMARY')
print('=' * 68)
print(f'\nEnvironment:')
print(f'  Python       : {sys.version.split()[0]}')
print(f'  scikit-learn : {sklearn.__version__}')
print(f'  xgboost      : {xgboost.__version__}')
print(f'  Random state : {RANDOM_STATE}')
print(f'  CV           : StratifiedKFold(n_splits=5, shuffle=True, random_state=42)')
print(f'\nDataset:')
print(f'  X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'  Features: {X_train.shape[1]}')
print(f'  Class 0: {CLASS_DIST["train_class0"]:,}  Class 1: {CLASS_DIST["train_class1"]:,}  Ratio: {CLASS_DIST["imbalance_ratio"]:.2f}:1')
print(f'\nCV Results (Best PR-AUC):')
for name, (sc, sd) in cv_scores.items():
    print(f'  {name}: {sc:.4f} ± {sd:.4f}')
print(f'\nTest Set Results (threshold=0.5):')
for r in tuned_results:
    print(f'  {r["model_name"]}: PR-AUC={r["pr_auc"]:.4f}  ROC-AUC={r["roc_auc"]:.4f}  '
          f'F1={r["f1"]:.4f}  Recall={r["recall"]:.4f}')
print(f'\nOOF Threshold Analysis:')
print(f'  RF Default  : {best_thr_rf_def:.2f}')
print(f'  RF Balanced : {best_thr_rf_bal:.2f}')
print(f'  XGBoost     : {best_thr_xgb:.2f}')
print(f'\n[SELECTED MODEL] {SELECTED_NAME}')
print(f'  PR-AUC   : {SELECTED_INFO["pr_auc"]:.4f}')
print(f'  ROC-AUC  : {SELECTED_INFO["roc_auc"]:.4f}')
print(f'  Threshold: {SELECTED_THR:.2f}')
print(f'\nLeakage Audit: {"PASS ✓" if leakage_pass else "FAIL ✗"}')
print(f'\nArtifacts:')
artifact_files = [
    os.path.join(MODELS_DIR, 'random_forest_tuned.pkl'),
    os.path.join(MODELS_DIR, 'random_forest_balanced_tuned.pkl'),
    os.path.join(MODELS_DIR, 'xgboost_tuned.pkl'),
    os.path.join(MODELS_DIR, 'final_candidate_model.pkl'),
    os.path.join(REPORTS_DIR, 'week8_cv_results.csv'),
    os.path.join(REPORTS_DIR, 'week8_tuning_results.csv'),
    os.path.join(REPORTS_DIR, 'week8_model_comparison.csv'),
    os.path.join(REPORTS_DIR, 'week8_model_selection_meta.json'),
    os.path.join(REPORTS_DIR, 'Week8_Tuning_Model_Selection_Report.md'),
]
for f in artifact_files:
    exists  = os.path.exists(f)
    size_mb = os.path.getsize(f)/1e6 if exists else 0
    fname   = os.path.relpath(f, PROJECT_ROOT)
    print(f'  [{"OK" if exists else "MISSING"}] {fname}  ({size_mb:.2f} MB)')

print(f'\nWEEK 8: COMPLETED ✓')
print(f'Next step: WEEK 9 — SHAP / EXPLAINABLE AI')
print('=' * 68)

WEEK 8 — FINAL STATUS SUMMARY

Environment:
  Python       : 3.14.4
  scikit-learn : 1.9.1
  xgboost      : 3.4.1
  Random state : 42
  CV           : StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Dataset:
  X_train: (23972, 33)  X_test: (5993, 33)
  Features: 33
  Class 0: 18,668  Class 1: 5,304  Ratio: 3.52:1

CV Results (Best PR-AUC):
  Random Forest (Default) Tuned: 0.5656 ± 0.0088
  Random Forest (Balanced) Tuned: 0.5627 ± 0.0096
  XGBoost Tuned: 0.5646 ± 0.0122

Test Set Results (threshold=0.5):
  Random Forest (Default) Tuned: PR-AUC=0.5610  ROC-AUC=0.7773  F1=0.4702  Recall=0.3597
  Random Forest (Balanced) Tuned: PR-AUC=0.5584  ROC-AUC=0.7753  F1=0.5408  Recall=0.5950
  XGBoost Tuned: PR-AUC=0.5620  ROC-AUC=0.7769  F1=0.5409  Recall=0.6161

OOF Threshold Analysis:
  RF Default  : 0.30
  RF Balanced : 0.55
  XGBoost     : 0.60

[SELECTED MODEL] XGBoost Tuned
  PR-AUC   : 0.5620
  ROC-AUC  : 0.7769
  Threshold: 0.60

Leakage Audit: PASS ✓

Artifacts:
  [OK] models\